In [ ]:
from imageio.v3 import imread, imwrite
from lumicks import pylake

import csv
import matplotlib.pyplot as plt
import numpy as np
import os

In [10]:
channel = 0
force_points = range(5,70,5)
force_range = 5
path = '/Users/sc13967/Library/CloudStorage/OneDrive-UniversityofBristol/People/Gemma Fisher/2026-06-11 Quantification of cohesin retention versus applied lateral force'

In [12]:
fig = plt.figure()

file_count = 0
for filename in os.listdir(path):
    if os.path.splitext(filename)[1] != ".h5":
        file_count += 1

file_idx = 0
for filename in os.listdir(path):
    if os.path.splitext(filename)[1] != ".h5":
        continue
    
    print(f"Processing: {filename}")
    
    # Reading .h5 file ahd kymograph    
    pylake_file = pylake.File(f"{path}/{filename}")
    kymo = next(iter(pylake_file.kymos.values()))
    
    # Getting forces
    line_timestamp_ranges = kymo.line_timestamp_ranges()
    forces = pylake_file.force2x.downsampled_over(line_timestamp_ranges).data
    print(forces)
    sdfsdfsf
    # Getting kymograph image
    kymo_image = kymo.get_image()[:,:,channel]
    
    # Loading kymograph mask
    mask_image = imread(f"{path}/{filename[:-3]}_kymograph0_mask.tif")
    active_timepoints = [np.sum(mask_image, axis=0) > 0]
    
    # For each specified force range, getting the pixels within the mask
    # Also, storing a labelled mask showing the pixels used
    intensities = {}
    label_mask = np.zeros(mask_image.shape)
    first_force = 0
    colour = plt.cm.tab10(file_idx)
    for i, force_point in enumerate(force_points):        
        curr_mask = mask_image.copy()
        curr_mask[:,forces<force_point-force_range/2] = 0
        curr_mask[:,forces>=force_point+force_range/2] = 0
        curr_mask[:,not active_timepoints] = 0
        label_mask[curr_mask>0] = i+1
        
        mean_intensity = np.mean(kymo_image[curr_mask>0])
        
        if i == 0:
            first_force = mean_intensity
        
        norm_intensity = 100*mean_intensity/first_force
            
        intensities[force_point] = [mean_intensity,norm_intensity, force_point-force_range/2, force_point+force_range/2]
    
        plt.plot(force_point, norm_intensity, "o", color=colour)
      
    imwrite(f"{path}/{filename[:-3]}_kymograph0_label_mask.tif", label_mask)
    
    with open(f"{path}/{filename[:-3]}_kymograph0_intensities.csv", 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Force","Intensity","Relative intensity (%)","Min force (pN)", "Max force (pN)"])
        for force, intensity in intensities.items():
            writer.writerow([force,intensity[0],intensity[1],intensity[2],intensity[3]])
            
    file_idx += 1

plt.ylim(0,140)
plt.xlabel("Force (pN)")
plt.ylabel("Relative intensity (%)")    
plt.savefig(f"{path}/Intensity_vs_force.png")
plt.savefig(f"{path}/Intensity_vs_force.svg")
    

Processing: 20240819-150426 Kymograph 5.h5
[ 2.47997977  2.45940696  2.44187021  2.53441059  2.37079364  2.54066633
  2.56939836  3.09381489  3.79201804  4.62253929  5.32614803  6.38716205
  7.68238863  9.00682204 10.69788851 12.58521129 14.71046388 16.90037197
 19.11687388 21.40472267 23.73160214 26.13251509 28.52699815 30.92072653
 33.20125983 35.4765594  37.54090841 39.69420155 41.90862652 43.66879755
 45.19271349 46.83219828 48.21005031 49.40764423 50.7738593  51.81808578
 52.90394777 53.72105119 54.58597555 55.38855565 56.16273042 56.90403984
 57.14125861 57.63662593 58.13336862 58.74077791 59.23732005 59.7301554
 59.76607517 59.92942945 59.98558345 60.28192726 60.18540166 59.81774742
 60.10044066 60.55105702 60.8592584  61.26910876 61.39825533 61.67363646
 61.75894477 62.01865429 61.82287517 61.77538155 61.96917654 62.07028245
 62.13910563 61.70974702 61.88975371 61.69515394 61.77063631 61.58177
 61.92082019 61.99726601 61.63361041 61.52835832 60.61166951 61.05651878
 61.27817857

NameError: name 'sdfsdfsf' is not defined

<Figure size 640x480 with 0 Axes>